# Weekly Lakehouse Table Maintenance — VACUUM

Removes unreferenced Parquet files (zombie files) left behind by overwrites, OPTIMIZE
compaction, and Delta time-travel across all Delta tables in a given lakehouse.

Unlike the Fabric Warehouse, the Lakehouse never auto-vacuums — old file versions
accumulate indefinitely until VACUUM is explicitly run.

Retention: 168 hours (7 days) minimum. Going below this risks reader failures or
table corruption if concurrent processes still reference older file versions.

Docs:
- https://learn.microsoft.com/en-us/fabric/data-engineering/lakehouse-table-maintenance
- https://learn.microsoft.com/en-us/fabric/data-engineering/delta-optimization-and-v-order
- https://learn.microsoft.com/en-us/fabric/fundamentals/table-maintenance-optimization

## Parameters

Set the target workspace and lakehouse by name. Retention defaults to 168 hours (7 days) —
do not lower this in production as Delta uses the retention window to protect files that
may still be referenced by concurrent readers.

Set `dry_run = True` to preview eligible files without deleting anything.

In [ ]:
# Pass workspace and lakehouse by name — no need to look up GUIDs manually.
# Leave either blank ("") to default to the workspace/lakehouse attached to this notebook.
workspace_name  = ""    # e.g. "My Fabric Workspace"
lakehouse_name  = ""    # e.g. "gold_lakehouse"
retention_hours = 168   # Minimum recommended: 168 (7 days). Do not lower in production.
dry_run         = False # Set True to preview what would be deleted without removing anything

## Setup

Resolves workspace and lakehouse names to GUIDs using `sempy.fabric` and validates the
retention threshold before proceeding. The safety guard raises an error if
`retention_hours` is below 168 and `dry_run` is False, preventing accidental data loss.

If `workspace_name` or `lakehouse_name` are left blank, the notebook defaults to the
workspace and lakehouse it is attached to.

In [ ]:
import sempy.fabric as fabric
from delta.tables import DeltaTable
from notebookutils import mssparkutils
from pyspark.sql import SparkSession
from datetime import datetime

spark = SparkSession.builder.getOrCreate()

# Safety guard — prevent accidental low-retention runs
if retention_hours < 168 and not dry_run:
    raise ValueError(
        f"retention_hours is set to {retention_hours}, which is below the 7-day (168 hour) "
        "minimum. This risks reader failures and table corruption. Set dry_run=True to preview, "
        "or increase retention_hours to at least 168."
    )

# Resolve workspace
if workspace_name:
    workspace_id = fabric.resolve_workspace_id(workspace_name)
else:
    workspace_id   = fabric.get_workspace_id()
    workspace_name = fabric.resolve_workspace_name(workspace_id)

# Resolve lakehouse ID by name within the workspace
if lakehouse_name:
    items_df = fabric.list_items(workspace=workspace_id, item_type="Lakehouse")
    matched  = items_df[items_df["Display Name"] == lakehouse_name]
    if matched.empty:
        raise ValueError(f"Lakehouse '{lakehouse_name}' not found in workspace '{workspace_name}'")
    lakehouse_id = matched["Id"].iloc[0]
else:
    lakehouse_id   = fabric.get_lakehouse_id()
    lakehouse_name = fabric.resolve_item_name(lakehouse_id, workspace=workspace_id)

base_path = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Tables"

print(f"Workspace       : {workspace_name} ({workspace_id})")
print(f"Lakehouse       : {lakehouse_name} ({lakehouse_id})")
print(f"Base path       : {base_path}")
print(f"Retention hours : {retention_hours} ({retention_hours / 24:.1f} days)")
print(f"Dry run         : {dry_run}")
print(f"Run started     : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## Discover Tables

Lists all table folders in the lakehouse `/Tables` directory. Same approach as the
configure table properties notebook — `mssparkutils.fs.ls` is used over the SQL catalog
for reliability across all table creation methods. Shortcut tables are excluded
automatically since their files are not owned by this lakehouse.

In [ ]:
table_items = mssparkutils.fs.ls(base_path)

table_paths = [
    item for item in table_items
    if item.isDir
    and not item.name.startswith("_")
    and not item.name.startswith(".")
]

print(f"Found {len(table_paths)} tables:")
for t in table_paths:
    print(f"  - {t.name}")

## Health Assessment

Before running VACUUM, assesses the fragmentation state of each table by reading its
Delta log statistics. This provides useful context in the summary — particularly for
identifying tables where active file fragmentation exists that VACUUM alone cannot fix.

Two flags are applied:
- `FRAGMENTED` — average file size below 25 MB, indicating small file accumulation that
  OPTIMIZE should address
- `OVERSIZED` — average file size above 2 GB, which can cause memory pressure during reads

Note: VACUUM removes unreferenced files only. It does not consolidate active fragmented
files — that requires OPTIMIZE.

In [ ]:
health_results = []

for item in table_paths:
    table_name = item.name
    table_path = f"{base_path}/{table_name}"

    try:
        if not DeltaTable.isDeltaTable(spark, table_path):
            health_results.append({
                "table":       table_name,
                "size_gb":     None,
                "num_files":   None,
                "avg_file_mb": None,
                "health_flag": "NOT DELTA"
            })
            continue

        details     = spark.sql(f"DESCRIBE DETAIL delta.`{table_path}`").collect()[0]
        size_bytes  = details["sizeInBytes"] or 0
        num_files   = details["numFiles"] or 0
        size_gb     = size_bytes / (1024 ** 3)
        avg_file_mb = (size_bytes / num_files / (1024 ** 2)) if num_files > 0 else 0

        # Flag tables with obvious file health issues
        if avg_file_mb < 25 and num_files > 10:
            health_flag = "FRAGMENTED"
        elif avg_file_mb > 2048:
            health_flag = "OVERSIZED FILES"
        else:
            health_flag = "OK"

        health_results.append({
            "table":       table_name,
            "size_gb":     round(size_gb, 3),
            "num_files":   num_files,
            "avg_file_mb": round(avg_file_mb, 2),
            "health_flag": health_flag
        })

        print(f"{table_name:40s} | {size_gb:6.2f} GB | {num_files:5d} files | {avg_file_mb:7.2f} MB avg | {health_flag}")

    except Exception as e:
        print(f"[WARN] Could not assess {table_name}: {e}")
        health_results.append({
            "table":       table_name,
            "size_gb":     None,
            "num_files":   None,
            "avg_file_mb": None,
            "health_flag": f"ERROR: {e}"
        })

## Run VACUUM

Loops through each discovered table and runs `VACUUM` with the configured retention window.

In dry run mode, Delta's built-in dry run returns the list of files that would be deleted
without removing anything. In live mode, any file not referenced by the Delta log within
the retention window is permanently deleted.

Each table is processed independently — an error on one table does not stop the run.

In [ ]:
vacuum_results = []

for item in table_paths:
    table_name = item.name
    table_path = f"{base_path}/{table_name}"
    result = {
        "table":          table_name,
        "status":         None,
        "files_eligible": None,
        "error":          None
    }

    try:
        if not DeltaTable.isDeltaTable(spark, table_path):
            result["status"] = "SKIPPED"
            result["error"]  = "Not a Delta table"
            vacuum_results.append(result)
            print(f"[SKIPPED] {table_name} — not a Delta table")
            continue

        if dry_run:
            # DRY RUN: Delta returns a dataframe of files that would be deleted
            # Count them to give a useful preview without deleting anything
            eligible_df    = spark.sql(f"VACUUM delta.`{table_path}` RETAIN {retention_hours} HOURS DRY RUN")
            files_eligible = eligible_df.count()
            result["status"]         = "DRY RUN"
            result["files_eligible"] = files_eligible
            print(f"[DRY RUN] {table_name} — {files_eligible} files eligible for deletion")
        else:
            spark.sql(f"VACUUM delta.`{table_path}` RETAIN {retention_hours} HOURS")
            result["status"] = "SUCCESS"
            print(f"[SUCCESS] {table_name} — vacuumed (retention: {retention_hours}h)")

    except Exception as e:
        result["status"] = "ERROR"
        result["error"]  = str(e)
        print(f"[ERROR] {table_name} — {e}")

    vacuum_results.append(result)

## Summary

Merges the health assessment and VACUUM outcomes into a single table for review.
Use this to confirm all tables were processed and to identify any that remain fragmented
after VACUUM — those tables likely need a manual OPTIMIZE run followed by another VACUUM
cycle to fully clean up.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, FloatType, IntegerType

# Merge health diagnostics and vacuum outcomes into a single summary table
health_map = {r["table"]: r for r in health_results}
vacuum_map = {r["table"]: r for r in vacuum_results}
all_tables = sorted(set(list(health_map.keys()) + list(vacuum_map.keys())))

summary_schema = StructType([
    StructField("table",          StringType(),  True),
    StructField("size_gb",        FloatType(),   True),
    StructField("num_files",      IntegerType(), True),
    StructField("avg_file_mb",    FloatType(),   True),
    StructField("health_flag",    StringType(),  True),
    StructField("vacuum_status",  StringType(),  True),
    StructField("files_eligible", StringType(),  True),
    StructField("error",          StringType(),  True),
])

summary_rows = []
for table in all_tables:
    h = health_map.get(table, {})
    v = vacuum_map.get(table, {})
    summary_rows.append((
        table,
        float(h["size_gb"])     if h.get("size_gb")     is not None else None,
        int(h["num_files"])     if h.get("num_files")   is not None else None,
        float(h["avg_file_mb"]) if h.get("avg_file_mb") is not None else None,
        h.get("health_flag", ""),
        v.get("status", ""),
        str(v["files_eligible"]) if v.get("files_eligible") is not None else "",
        v.get("error") or ""
    ))

summary_df = spark.createDataFrame(summary_rows, schema=summary_schema)

print(f"\n{'='*60}")
print(f"RUN COMPLETE — {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Workspace : {workspace_name}")
print(f"Lakehouse : {lakehouse_name}")
print(f"{'='*60}")
print(f"  Total tables     : {len(all_tables)}")
print(f"  Vacuumed         : {sum(1 for r in vacuum_results if r['status'] == 'SUCCESS')}")
print(f"  Dry run          : {sum(1 for r in vacuum_results if r['status'] == 'DRY RUN')}")
print(f"  Skipped          : {sum(1 for r in vacuum_results if r['status'] == 'SKIPPED')}")
print(f"  Errors           : {sum(1 for r in vacuum_results if r['status'] == 'ERROR')}")
print(f"  Fragmented tables: {sum(1 for r in health_results if r.get('health_flag') == 'FRAGMENTED')}")

display(summary_df)